# s13 — Per-neuron segregation index (MCNS)

MCNS analogue of the FAFB `Data_Processing/s13_segregation_index.ipynb`.

For each right-hemisphere group (FFP / FBP / BDP / Others), this notebook reads the
root_id list saved by `MCNS/Figures/fig_1_postPI_prePI_right_neurons.m`
(`MCNS/Processed_Data/right_<group>_root_ids.csv`, no header, root_id only), fetches
each neuron's skeleton and synapses from neuprint (`male-cns:v0.9`), splits it into
axon/dendrite, and computes its segregation index. The results are written to
`MCNS/Processed_Data/right_<group>_segregation_index.csv` (columns: `root_id`,
`segregation_index`).

Requires `neuprint-python`, `navis`, and a neuprint auth token (set
`NEUPRINT_APPLICATION_CREDENTIALS` or place it in `MCNS/Data_Processing/.neuprint_token`).

In [ ]:
import os
import navis
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
from tqdm import tqdm
from neuprint import Client, NeuronCriteria as NC, SynapseCriteria as SC, fetch_neurons, fetch_skeleton, fetch_synapses

In [ ]:
from pathlib import Path

SERVER = "neuprint.janelia.org"
DATASET = "male-cns:v0.9"

# MCNS project folders. baseDir (the MCNS root) is resolved from the notebook's
# location (run from MCNS/Data_Processing/), so the code runs after download
# without editing any path.
baseDir = Path.cwd().parent
processed_dir = str(baseDir / "Processed_Data")
TOKEN_FILE = baseDir / "Data_Processing" / ".neuprint_token"


def load_token(token_file: Path = TOKEN_FILE) -> str:
    token = os.environ.get("NEUPRINT_APPLICATION_CREDENTIALS", "").strip()
    if token:
        return token

    token_file = Path(token_file)
    if token_file.exists():
        token = token_file.read_text(encoding="utf-8").strip()
        if token:
            return token

    raise RuntimeError(
        "No neuprint token found. Set NEUPRINT_APPLICATION_CREDENTIALS "
        f"or place the token in {token_file}."
    )


client = Client(SERVER, dataset=DATASET, token=load_token())
print(f"dataset: {DATASET}")

In [ ]:
def fetch_navis_neuron(body_id: int) -> navis.TreeNeuron:
    """Fetch a neuron's skeleton + synapses from neuprint and return a healed navis TreeNeuron."""
    body_id = int(body_id)

    skeleton_df = fetch_skeleton(body_id, format="pandas", heal=False, client=client)
    if skeleton_df is None or skeleton_df.empty:
        raise ValueError(f"{body_id}: skeleton not found in {DATASET}")

    nodes = (
        skeleton_df.rename(columns={"rowId": "node_id", "link": "parent_id"})
        [["node_id", "x", "y", "z", "radius", "parent_id"]]
        .copy()
    )
    nodes[["node_id", "parent_id"]] = nodes[["node_id", "parent_id"]].astype(np.int64)
    neuron = navis.TreeNeuron(nodes, id=body_id)

    syn_df = fetch_synapses(
        NC(bodyId=body_id, client=client),
        SC(primary_only=True, client=client),
        client=client,
    ).copy()

    if not syn_df.empty:
        node_tree = cKDTree(nodes[["x", "y", "z"]].to_numpy(dtype=float))
        _, node_idx = node_tree.query(syn_df[["x", "y", "z"]].to_numpy(dtype=float), k=1)

        connectors = syn_df[["type", "x", "y", "z", "roi", "confidence"]].copy()
        connectors.insert(0, "connector_id", np.arange(1, len(connectors) + 1, dtype=np.int64))
        connectors.insert(1, "node_id", nodes.iloc[node_idx]["node_id"].to_numpy())
        neuron.connectors = connectors

    neuron = navis.heal_skeleton(neuron)

    meta_df = fetch_neurons(NC(bodyId=body_id, client=client), omit_rois=True, client=client)
    if not meta_df.empty:
        soma_loc = meta_df.iloc[0].get("somaLocation")
        if isinstance(soma_loc, (list, tuple, np.ndarray)) and len(soma_loc) == 3:
            soma_tree = cKDTree(neuron.nodes[["x", "y", "z"]].to_numpy(dtype=float))
            _, soma_idx = soma_tree.query(np.asarray(soma_loc, dtype=float), k=1)
            neuron.soma = int(neuron.nodes.iloc[int(soma_idx)]["node_id"])

    return neuron


def si_from_root_id(root_id: int) -> float:
    """Split a neuron into axon/dendrite and return its segregation index."""
    # The CSV root_id column is treated as a neuprint bodyId.
    body_id = int(root_id)
    neuron = fetch_navis_neuron(body_id)
    reroot_soma = getattr(neuron, "soma", None) is not None
    split = navis.split_axon_dendrite(
        neuron,
        metric="segregation_index",
        reroot_soma=reroot_soma,
    )
    return float(navis.segregation_index(split))

In [ ]:
# Compute the segregation index for each group and save root_id + segregation_index.
groups = {
    "FFP": "right_FFP_root_ids.csv",
    "FBP": "right_FBP_root_ids.csv",
    "BDP": "right_BDP_root_ids.csv",
    "Others": "right_Others_root_ids.csv",
}

for group, in_name in groups.items():
    in_path = os.path.join(processed_dir, in_name)
    df = pd.read_csv(in_path, header=None, names=["root_id"])   # no header, root_id only

    segregation_index = []
    errors = []
    root_ids = df["root_id"].tolist()
    n_total = len(root_ids)

    for i, rid in enumerate(tqdm(root_ids, desc=group), start=1):
        percent = (i / n_total) * 100 if n_total else 0.0
        try:
            si_val = si_from_root_id(rid)
            val = round(si_val, 2)
            print(f"[{group} {i}/{n_total} | {percent:.1f}%] {int(rid)}: SI = {si_val:.3f}")
        except Exception as e:
            val = np.nan
            errors.append((rid, str(e)))
            print(f"[{group} {i}/{n_total} | {percent:.1f}%] {rid}: failed -> {e}")

        segregation_index.append(val)

    df["segregation_index"] = segregation_index
    out_path = os.path.join(processed_dir, f"right_{group}_segregation_index.csv")
    df.to_csv(out_path, index=False)

    n_fail = int(np.isnan(np.array(segregation_index, dtype=float)).sum())
    print(f"{group}: {n_total} total, {n_fail} failed -> saved {out_path}")